# 15 — Evals, traces, cost

A risk function will not ask whether the demo looked good. It will ask three things:

1. Did it do the job?
2. What did it actually do?
3. What did it cost?

You already have the pieces. Module 00 turned `usage` into dollars. Module 03 printed the loop. Module 05 treated the message list as a budget. Today they sit on **one Chinook desk**: a ledger per turn, the message list as a trace, a checker against named facts.

LangSmith and Langfuse are names on a slide. We do not create an account. Module 08 already printed an OpenAI traces URL. That product is this list, drawn as a timeline.

```mermaid
graph TD
    A[Question] --> B[Official loop]
    B --> C[Ledger: tokens and dollars per turn]
    B --> D[Trace: the messages list]
    B --> E[Final sentence]
    E --> F[Checker]
    F --> G[Pass or fail]
```


## 1. Learn

```
00  usage, once, for one call
03  the official loop
05  the list is a budget
08  a hosted trace URL
13  one agent, three Chinook tools
15  you are here — score it, show the path, print the bill
```

Five dimensions, not one score. Same five as the slides. Four of them are in this notebook; latency is named, not measured.

| Dimension | What you check | What it is not |
|---|---|---|
| Task completion | The sentence contains the facts | A thumbs-up from the room |
| Tool-call correctness | The named tools actually ran | The model *said* it looked them up |
| Groundedness | The numbers in the sentence match the observations | A fluent paragraph |
| Cost per task | Dollars for this question | Tokens with no price |
| Latency | How long the loop took | Not today. The bill is the one that surprises people. |

A hosted tracer (OpenAI traces, LangSmith, Langfuse, Azure) stores the same list you are about to print. The lesson is the list, not the vendor. **Anything that needs a per-student signup is a failure mode in this room.** We stay on the class key.

A per-task **budget cap** belongs in the loop, next to the turn cap. Tokens without a dollar cap are how a retry storm becomes a finance ticket.

Cut first if the room is behind: the 1,000 / 100,000 multiplier. Keep the ledger.


## 2. Do

### Load Chinook, prices, three named tools

Same functions as 08 and 13. No free SQL. The model still does not touch the database. Prices come from `.env`, same keys as 00.


In [1]:
from pathlib import Path
import json
import os
import re
import sqlite3
import unicodedata

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


def fold(text: str) -> str:
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in nfkd if not unicodedata.combining(ch)).casefold()


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
price_in = float(os.environ["PRICE_INPUT_PER_MILLION"])
price_out = float(os.environ["PRICE_OUTPUT_PER_MILLION"])
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing from .env."

client = OpenAI()
db = sqlite3.connect(ROOT / "data" / "chinook.db", check_same_thread=False)
db.row_factory = sqlite3.Row

print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print(f"prices: ${price_in}/1M input, ${price_out}/1M output")
print("customers:", db.execute("select count(*) from customers").fetchone()[0])


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
prices: $0.2/1M input, $1.25/1M output
customers: 59


In [2]:
def find_customer(name: str):
    needle = fold(name).strip()
    rows = db.execute(
        "select CustomerId, FirstName, LastName, SupportRepId from customers"
    ).fetchall()
    hits = []
    for row in rows:
        full = fold(row["FirstName"] + " " + row["LastName"])
        if needle == full or needle == fold(row["FirstName"]) or needle in full:
            hits.append(row)
    if len(hits) == 1:
        return hits[0]
    if not hits:
        return None
    return "more than one customer matches that name"


def _customer_row(name: str):
    row = find_customer(name)
    if row is None:
        return None, "no customer by that name"
    if isinstance(row, str):
        return None, row
    return row, None


def count_invoices_fn(name: str) -> str:
    row, err = _customer_row(name)
    if err:
        return err
    n = db.execute(
        "select count(*) from invoices where CustomerId = ?",
        (row["CustomerId"],),
    ).fetchone()[0]
    return f"{row['FirstName']} {row['LastName']} has {n} invoices"


def invoice_total_fn(name: str) -> str:
    row, err = _customer_row(name)
    if err:
        return err
    total = db.execute(
        "select round(sum(Total), 2) from invoices where CustomerId = ?",
        (row["CustomerId"],),
    ).fetchone()[0]
    return f"{row['FirstName']} {row['LastName']} has spent {total} dollars"


def support_rep_fn(name: str) -> str:
    row, err = _customer_row(name)
    if err:
        return err
    rep = db.execute(
        "select FirstName || ' ' || LastName from employees where EmployeeId = ?",
        (row["SupportRepId"],),
    ).fetchone()[0]
    return f"{row['FirstName']} {row['LastName']}'s support rep is {rep}"


print(count_invoices_fn("Helena"))
print(invoice_total_fn("Helena Holy"))
print(support_rep_fn("Helena Holý"))


Helena Holý has 7 invoices
Helena Holý has spent 49.62 dollars
Helena Holý's support rep is Steve Johnson


Helena: **7**, **49.62**, **Steve Johnson**. No model yet. Those strings are what the checker will look for.

### Official loop with a ledger

Same `for` as 03. Each `create` already returns `usage`. We turn it into dollars and keep a running total. A budget cap sits next to the turn cap. We set it high so this question finishes. Point at the `if`.


In [3]:
def schema(tool_name, description, **props):
    return {"type": "function", "function": {
        "name": tool_name, "description": description,
        "parameters": {"type": "object", "properties": props, "required": list(props)}}}


tools = [
    schema("count_invoices", "How many invoices this customer has.",
           name={"type": "string"}),
    schema("invoice_total", "Total amount this customer has spent, in dollars.",
           name={"type": "string"}),
    schema("support_rep", "The support representative for this customer.",
           name={"type": "string"}),
]

HANDLERS = {
    "count_invoices": count_invoices_fn,
    "invoice_total": invoice_total_fn,
    "support_rep": support_rep_fn,
}


def dollars(prompt_tokens, completion_tokens):
    return (
        prompt_tokens / 1_000_000 * price_in
        + completion_tokens / 1_000_000 * price_out
    )


def run_tool(call):
    args = json.loads(call.function.arguments or "{}")
    arg = args.get("name", "")
    fn = HANDLERS[call.function.name]
    return call.function.name, arg, fn(arg)


SYSTEM = "Use the tools. Do not invent numbers or names."
BUDGET = 0.05


def run_loop(question, max_turns=5, budget=BUDGET):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    ledger = []
    tools_used = []
    observations = []
    final_text = None
    running = 0.0
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            max_completion_tokens=200,
            reasoning_effort="none",
        )
        message = response.choices[0].message
        u = response.usage
        cost = dollars(u.prompt_tokens, u.completion_tokens)
        running = running + cost
        names = [c.function.name for c in (message.tool_calls or [])]
        ledger.append({
            "turn": turn + 1,
            "finish": response.choices[0].finish_reason,
            "tools": names,
            "prompt_tokens": u.prompt_tokens,
            "completion_tokens": u.completion_tokens,
            "cost": cost,
            "running": running,
        })
        print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")
        print(
            f"  prompt={u.prompt_tokens}  completion={u.completion_tokens}  "
            f"cost=${cost:.6f}  running=${running:.6f}"
        )
        if running > budget:
            print("stopped: budget")
            break
        if not message.tool_calls:
            final_text = message.content
            print(final_text)
            break
        messages.append(message)
        for call in message.tool_calls:
            name, arg, result = run_tool(call)
            tools_used.append(name)
            observations.append(result)
            print(f"  {name}({arg!r}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    else:
        print("stopped: cap was", max_turns)
    return {
        "final_text": final_text,
        "ledger": ledger,
        "tools_used": tools_used,
        "observations": observations,
        "n_tools": len(tools_used),
        "prompt_tokens": sum(row["prompt_tokens"] for row in ledger),
        "cost": running,
        "messages": messages,
    }


QUESTION = (
    "How many invoices does Helena Holý have, what did she spend, "
    "and who is her support rep?"
)
helena = run_loop(QUESTION)
print()
print("n_tools:", helena["n_tools"])
print("cost:   ", f"${helena['cost']:.6f}")


--- turn 1 finish_reason: tool_calls ---
  prompt=206  completion=71  cost=$0.000130  running=$0.000130
  count_invoices('Helena Holý') -> Helena Holý has 7 invoices
  invoice_total('Helena Holý') -> Helena Holý has spent 49.62 dollars
  support_rep('Helena Holý') -> Helena Holý's support rep is Steve Johnson
--- turn 2 finish_reason: stop ---
  prompt=325  completion=30  cost=$0.000102  running=$0.000232
Helena Holý has **7 invoices**, has spent **$49.62**, and her support rep is **Steve Johnson**.

n_tools: 3
cost:    $0.000232


Cover the ledger. Turn 1 `prompt_tokens` is the system prompt, the schemas, and the question. The next turn is that plus the observations. That climb is module 05, now billed.

### The trace is the list

A hosted tracer stores this. We print it.


In [4]:
def show_trace(messages):
    for m in messages:
        if isinstance(m, dict):
            role = m["role"]
            if role == "tool":
                print(f"{'tool':<12}{m['content']}")
            else:
                text = (m.get("content") or "").replace("\n", " ")
                print(f"{role:<12}{text[:120]}")
            continue
        if getattr(m, "tool_calls", None):
            for c in m.tool_calls:
                print(f"{'assistant':<12}{c.function.name}({c.function.arguments})")
        else:
            print(f"{'assistant':<12}{(m.content or '')[:120]}")


show_trace(helena["messages"])


system      Use the tools. Do not invent numbers or names.
user        How many invoices does Helena Holý have, what did she spend, and who is her support rep?
assistant   count_invoices({"name": "Helena Holý"})
assistant   invoice_total({"name": "Helena Holý"})
assistant   support_rep({"name": "Helena Holý"})
tool        Helena Holý has 7 invoices
tool        Helena Holý has spent 49.62 dollars
tool        Helena Holý's support rep is Steve Johnson


If a later tool could pay someone, this list is the audit. You do not get it for free from a thumbs-up.

### A checker, not a vibe

`score` is a regex you wrote. Task completion looks at the sentence. Tool-call correctness looks at `tools_used`. Groundedness checks that the numbers in the sentence also appeared in the observations.


In [5]:
def score(text, patterns):
    t = str(text or "")
    return all(re.search(p, t, re.I) for p in patterns)


text = helena["final_text"]
obs = " ".join(helena["observations"])
task = score(text, [r"\b7\b", r"49\.62", r"steve"])
tools_ok = set(helena["tools_used"]) >= {
    "count_invoices", "invoice_total", "support_rep",
}
grounded = score(text, [r"\b7\b", r"49\.62"]) and score(obs, [r"\b7\b", r"49\.62"])

print("task completion:   ", task)
print("tool-call correct: ", tools_ok)
print("grounded:          ", grounded)
print("n_tools:           ", helena["n_tools"])
print("prompt_tokens:     ", helena["prompt_tokens"])
print("cost:              ", f"${helena['cost']:.6f}")


task completion:    True
tool-call correct:  True
grounded:           True
n_tools:            3
prompt_tokens:      531
cost:               $0.000232


Three booleans, three different questions. A fluent sentence that invents 8 invoices would pass a vibe check and fail all three.

### Ask for a shape, not a sentence

`score` reaches for a regex because a sentence is what we asked for. That is a choice, and it is the expensive one: every check downstream now depends on the model's prose keeping the same shape.

The alternative is to ask for the shape up front. `response_format` with a JSON schema makes the API enforce it, and `strict: True` means those fields, those types, and nothing extra. Same observations, one more call, and the checker stops being a regex.

This is the **structured output** row of the context table from module 05, and it is the cheapest reliability lever in the course.

In [ ]:
FACTS_SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_count": {"type": "integer"},
        "total_spend": {"type": "number"},
        "support_rep": {"type": "string"},
    },
    "required": ["invoice_count", "total_spend", "support_rep"],
    "additionalProperties": False,
}

shaped = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "Fill the schema from the observations. Do not invent values."},
        {"role": "user", "content": "\n".join(helena["observations"])},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "customer_facts", "schema": FACTS_SCHEMA, "strict": True},
    },
    max_completion_tokens=200,
    reasoning_effort="none",
)

facts = json.loads(shaped.choices[0].message.content)
print(facts)
print()
print("invoice_count == 7:   ", facts["invoice_count"] == 7)
print("total_spend == 49.62: ", facts["total_spend"] == 49.62)
print("support_rep is Steve: ", "Steve" in facts["support_rep"])

No regex. `facts["invoice_count"]` is an `int` and the check is `== 7`. The model also cannot add a fourth field: `additionalProperties: False` plus `strict: True` are enforced by the API, not by your prompt.

The cost is one extra call and a schema somebody has to maintain. The return is a checker that does not break when the model rephrases, and an output the next system in the chain can consume without parsing prose.

## 3. Observe

Same Helena run. The store did not change. The accounting did.

In [6]:
print(f"{'turn':<6}{'finish':<14}{'prompt':>8}{'compl':>8}{'cost':>12}  calls")
for row in helena["ledger"]:
    calls = ",".join(row["tools"]) or "-"
    print(
        f"{row['turn']:<6}{row['finish']:<14}"
        f"{row['prompt_tokens']:>8}{row['completion_tokens']:>8}"
        f"{row['cost']:>12.6f}  {calls}"
    )
print()
first = helena["ledger"][0]["prompt_tokens"]
last = helena["ledger"][-1]["prompt_tokens"]
print("turn 1 prompt_tokens:", first)
print("last  prompt_tokens:", last)
print("delta:              ", last - first)
print()
print(f"this task:            ${helena['cost']:.6f}")
print(f"1,000 of this task:   ${1000 * helena['cost']:.4f}")
print(f"100,000 of this task: ${100000 * helena['cost']:.2f}")


turn  finish          prompt   compl        cost  calls
1     tool_calls         206      71    0.000130  count_invoices,invoice_total,support_rep
2     stop               325      30    0.000102  -

turn 1 prompt_tokens: 206
last  prompt_tokens: 325
delta:               119

this task:            $0.000232
1,000 of this task:   $0.2324
100,000 of this task: $23.24


Things to notice:

- Prompt tokens **go up** after the first tool turn. Every observation rides along. That is why 05 existed, and why a one-file RAG hop is cheaper than a fishing expedition.
- The cents on this question are not the lesson. **1,000** and **100,000** of the same loop are. An app that retries, fans out, or greets every page load with an agent is spending `usage` in a tight loop.
- `BUDGET = 0.05` never fired. Good. The cap is still in the `for`. Take it out and the only backstop is `max_turns`.
- OpenAI traces / LangSmith / Langfuse would draw `show_trace` as a waterfall. They would not replace `score`. A pretty timeline of a wrong answer is still a wrong answer.
- Caching (cheaper input tokens on a repeated prefix) and model routing (nano for the desk, mini for a specialist) are the next two levers. We will not add them today. The ledger is how you would know they helped.

## 4. Challenge

Same loop. A new customer:

> How many invoices does Puja Srivastava have, what did she spend, and who is her support rep?

Puja: **6**, **$36.64**, **Jane Peacock**.

Bind:

- `final_text` — the last sentence
- `cost` — dollars for this run, greater than zero
- `n_tools` — how many tools actually ran

The next cell checks the three facts, that you spent some money, and that at least two tools ran. It does not score the wording.


In [ ]:
# final_text, cost, n_tools = ...


In [ ]:
t = str(final_text)
assert re.search(r"\b6\b", t), "Puja has 6 invoices"
assert "36.64" in t, "Puja spent 36.64 dollars"
assert "Jane" in t, "Puja's support rep is Jane Peacock"
assert cost > 0, "cost should be dollars, greater than zero"
assert n_tools >= 2, "this question needs more than one tool"
print("looks good")
